# PSELDNets — v9パイロット（車=第5クラス、音声はv8をそのまま流用）

**目的（v9設計の先行検証、半日実験）**:
- Q1: 車の走行音を「音色」で判別できるか（5クラスでも取り違え≈0が保たれるか）
- Q2: 警告音と車が**同時に鳴っている**時間帯に、両方を方向付きで拾えるか

**仕組み**: 音声はv8の400本と同一。ラベルだけ「車」を第5クラスとして追加した
（車が背景雑音より大きく聞こえる瞬間のみ正解。方向誤差0.3°で検証済み。
50本は車が一度も聞こえない=車なしと答えるべき負例）。

## ⚠️ 使用前に必ず確認
1. ランタイム → T4 GPU
2. Driveの `PSELDNets_data/` に **`dataset_outdoor_siren_v8.zip`（既存）** と
   **`pilot_v9_labels.zip`（新規・0.2MB）** の両方があること
3. **このランタイムではv8の学習は行わない**（クラス辞書を5クラスに上書きするため）
4. fold3（test）は使わない（従来どおり）

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Drive マウントとパス設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ==== 設定 ====
DRIVE_DATA = '/content/drive/MyDrive/PSELDNets_data'
DRIVE_LOGS = '/content/drive/MyDrive/PSELDNets_logs'
DRIVE_CKPT = '/content/drive/MyDrive/PSELDNets_ckpts'
DATASET    = 'outdoor_siren_v9pilot'
EXP_NAME   = 'outdoor_siren_v9pilot_run1'

import os
for d in [DRIVE_DATA, DRIVE_LOGS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)
V8_ZIP = f'{DRIVE_DATA}/dataset_outdoor_siren_v8.zip'
PILOT_ZIP = f'{DRIVE_DATA}/pilot_v9_labels.zip'
assert os.path.exists(V8_ZIP), '⚠️ dataset_outdoor_siren_v8.zip がDriveにありません'
assert os.path.exists(PILOT_ZIP), '⚠️ pilot_v9_labels.zip をアップロードしてください'
print(f'OK: v8 zip {os.path.getsize(V8_ZIP)/1e6:.0f}MB / pilot zip {os.path.getsize(PILOT_ZIP)/1e6:.1f}MB')

## 3. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')

## 4. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。
ここでは**触らず**、不足しているものだけ追加します（v1-v4 と同一・再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

import numpy, lightning, torchmetrics, librosa
print(f'numpy {numpy.__version__} / lightning {lightning.__version__} / '
      f'torchmetrics {torchmetrics.__version__} / librosa {librosa.__version__}')

## 5. 事前学習チェックポイント（Drive キャッシュ → なければ HF から）

In [ ]:
import shutil

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'
CACHE = f'{DRIVE_CKPT}/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    if os.path.exists(CACHE):
        print('Drive キャッシュからコピー...')
        shutil.copy(CACHE, CKPT)
    else:
        print('HuggingFace からダウンロード...')
        from huggingface_hub import hf_hub_download
        src = hf_hub_download(repo_id='Jinbo-HU/PSELDNets',
                              filename='model/mACCDOA-HTSAT-0.567.ckpt',
                              repo_type='dataset')
        shutil.copy(src, CKPT)
        shutil.copy(CKPT, CACHE)   # 次回用に Drive へキャッシュ
print(f'OK: {CKPT} ({os.path.getsize(CKPT)/1e6:.0f} MB)')

## 6. データセット展開

zip は `datasets/...` 構成なのでリポジトリ直下で解凍するだけ。
クラス辞書 `cls_indices_train.tsv`（**本データセット専用の4クラス**: Siren/Horn/
BackupBeep/BikeBell）も同梱。

In [ ]:
import zipfile, os, shutil

# v8の音声を展開（このランタイムに無ければ）
if not os.path.exists('datasets/outdoor_siren_v8/foa'):
    with zipfile.ZipFile(V8_ZIP) as z:
        z.extractall('.')
    print('v8 unzipped')

# パイロットのラベル（5クラス辞書 + 車入りmetadata）を展開
# 注意: datasets/cls_indices_train.tsv が5クラス版に上書きされる
with zipfile.ZipFile(PILOT_ZIP) as z:
    z.extractall('.')
print('pilot labels unzipped')

# 音声はv8のものをそのまま流用（コピー）
if not os.path.exists(f'datasets/{DATASET}/foa'):
    shutil.copytree('datasets/outdoor_siren_v8/foa', f'datasets/{DATASET}/foa')
    print('foa copied from v8')

n_foa = len(os.listdir(f'datasets/{DATASET}/foa'))
n_meta = len(os.listdir(f'datasets/{DATASET}/metadata'))
n_cls = len(open('datasets/cls_indices_train.tsv').readlines())
print(f'foa: {n_foa} / metadata: {n_meta} / classes: {n_cls}')
assert n_foa == 400 and n_meta == 400 and n_cls == 5, '⚠️ ファイル数が想定と違います'

## 7. 設定ファイル2つを作成（新規追加のみ・リポジトリ既存ファイルは無編集）

In [ ]:
data_yaml = """audio_type: foa
audio_feature: logmelIV
sample_rate: 24000
nfft: 1024
n_mels: 64
hoplen: 240
window: hann

train_chunklen_sec: 10
train_hoplen_sec: 10
test_chunklen_sec: 10
test_hoplen_sec: 10

train_dataset:
  outdoor_siren_v9pilot: [fold1_room1]
valid_dataset:
  outdoor_siren_v9pilot: [fold2_room1]
test_dataset:
  outdoor_siren_v9pilot: [fold3_room1]
"""

data_yaml_valinfer = data_yaml.replace(
    'test_dataset:\n  outdoor_siren_v9pilot: [fold3_room1]',
    'test_dataset:\n  outdoor_siren_v9pilot: [fold2_room1]')

exp_yaml = """# @package _global_
defaults:
 - override /data: outdoor_siren_v9pilot.yaml
 - override /loss: multi_accdoa.yaml
 - _self_

task_name: outdoor_siren_v9pilot

model:
  batch_size: 8
  kwargs:
    pretrained_path: ckpts/mACCDOA-HTSAT-0.567.ckpt
    audioset_pretrain: false
  optimizer:
    kwargs: {lr: 0.0003}
  lr_scheduler:
    kwargs: {step_size: 60}

trainer:
  max_epochs: 100
  check_val_every_n_epoch: 5
"""

exp_yaml_valinfer = exp_yaml.replace('override /data: outdoor_siren_v9pilot.yaml',
                                     'override /data: outdoor_siren_v9pilot_valinfer.yaml')

open('configs/data/outdoor_siren_v9pilot.yaml', 'w').write(data_yaml)
open('configs/data/outdoor_siren_v9pilot_valinfer.yaml', 'w').write(data_yaml_valinfer)
open('configs/experiment/outdoor_siren_v9pilot.yaml', 'w').write(exp_yaml)
open('configs/experiment/outdoor_siren_v9pilot_valinfer.yaml', 'w').write(exp_yaml_valinfer)
print('wrote configs (v9pilot + valinfer)')

## 8. 前処理（ラベル → HDF5、クリップ索引の作成。1分未満）

In [ ]:
IDX = f'_hdf5/data/24000fs/wav/dev/{DATASET}_10sChunklen_10sHoplen_train.csv'
if not os.path.exists(IDX):
    !python src/preproc.py dataset={DATASET}
else:
    print('既に前処理済み')
!head -3 {IDX}

## 9. 実行前チェック

In [ ]:
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt',     'チェックポイント'),
    ('datasets/cls_indices_train.tsv',      'クラス辞書 TSV (5クラス)'),
    (f'datasets/{DATASET}/foa',             'FOA データ (400)'),
    (f'datasets/{DATASET}/metadata',        'ラベル CSV (400)'),
    (f'configs/experiment/{DATASET}.yaml',  '実験設定'),
    (IDX,                                   'クリップ索引'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

## 10. 学習（T4 で 60〜90 分見込み、100epoch）

- 学習・ckpt選択に使うのは train(fold1) と val(fold2) のみ。test(fold3) は触らない
- `last.ckpt` があれば自動再開。データを変えたら `EXP_NAME` を新名に
- epoch数100はv6/v7と同じ根拠（train240本、v6はep45以降収束）

In [ ]:
LAST = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/last.ckpt'
resume = f'ckpt_path={LAST}' if os.path.exists(LAST) else ''
print('resume:', resume or '(new run)')

!python src/train.py experiment={DATASET} \
    experiment_name={EXP_NAME} \
    paths.log_dir={DRIVE_LOGS} \
    {resume}

## 11. 結果の確認（val基準）

パイロットの判定基準（数値そのものよりQ1/Q2の答えを見る）:
- **Q1**: 学習後のval解剖で substitution（車↔警告音の取り違え）がほぼ0か
- **Q2**: 警告音と車が同時の時間帯に両方を検出・定位できるか（ローカルのstep8で解剖）
- 参考: v8 run1（4クラス、同一音声）は val SELD 0.052。車クラスが増えた分の
  変動は想定内。数値の絶対比較はしない

In [ ]:
import re

log_path = f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/train.log'
lines = [l for l in open(log_path, errors='ignore')
         if 'val/macro' in l or 'train: loss_all' in l]
print(f'--- {log_path} ---')
for l in lines:
    print(re.sub(r'\x1b\[[0-9;]*m', '', l).rstrip())

vals = [l for l in lines if 'val/macro' in l]
if vals:
    print('\n=== 最終 val/macro ===')
    print(re.sub(r'\x1b\[[0-9;]*m', '', vals[-1]).strip())

---
## メモ

- 音声はv8と同一。ラベルのみ拡張（車=クラス4、「聞こえる瞬間」ルール）
- ラベル生成・検証はローカル `scripts/step9_pilot_car_labels.py`（方向誤差0.3°、
  可聴率とSNR/SIRの相関が物理と整合、をローカルで確認済み）
- 学習後: セル12（val推論→連結CSV）まで実行 → ローカルで
  `step8_error_anatomy_mc.py --pred out/predictions_v9pilot_val --ds out/dataset_outdoor_siren_v9pilot`
- **このランタイムでv8を学習しないこと**（クラス辞書が5クラスに上書きされているため）

## 12. 開発用推論（val=fold2、誤り解剖・イベント指標用）

学習後に実行。予測CSVを連結してDriveに保存 → ローカルでstep8/step8dに掛ける。
（fold3への推論は卒論の最終数値を出すときに1回だけ。そのときは
`experiment=outdoor_siren_v9pilot mode=test` で実行する）

In [ ]:
import glob, os
best_ckpt = sorted(glob.glob(f'{DRIVE_LOGS}/{DATASET}/runs/{EXP_NAME}/checkpoints/epoch_*.ckpt'))[-1]
print('using:', best_ckpt)

!python src/infer.py experiment=outdoor_siren_v9pilot_valinfer \
    mode=test \
    ckpt_path="{best_ckpt}" \
    model.kwargs.pretrained_path=null \
    experiment_name=infer_{EXP_NAME}_val \
    paths.log_dir={DRIVE_LOGS}

# 予測CSVを1ファイルに連結してDriveへ（ローカル取得の省力化）
exp = f'infer_{EXP_NAME}_val'
sub = f'{DRIVE_LOGS}/{DATASET}/runs/{exp}/submissions'
out_lines = []
for p in sorted(glob.glob(f'{sub}/*.csv')):
    stem = os.path.basename(p)[:-4]
    for line in open(p):
        if line.strip():
            out_lines.append(f'{stem},{line.strip()}')
out = f'/content/drive/MyDrive/PSELDNets_data/{exp}_all.csv'
open(out, 'w').write('\n'.join(out_lines))
print('wrote', out, len(out_lines), 'lines')